In [24]:
!pip install textblob


  Using cached textblob-0.19.0-py3-none-any.whl.metadata (4.4 kB)
Using cached textblob-0.19.0-py3-none-any.whl (624 kB)



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [25]:
import nltk
nltk.download('punkt')


[nltk_data] Downloading package punkt to C:\Users\Jeffrin
[nltk_data]     Jessika\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt.zip.


True

PlayStore Data Analytics

#setup

In [ ]:
# =========================
# BASE SETUP CELL
# =========================

import pandas as pd
import numpy as np
import plotly.express as px

# Load Play Store dataset
df = pd.read_csv("Play Store Data.csv")
playstore_df = pd.read_csv("Play Store Data.csv")


# Quick check
df.head()


,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver
0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,4.1,159,19M,"10,000+",Free,0,Everyone,Art & Design,"January 7, 2018",1.0.0,4.0.3 and up
1,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,"500,000+",Free,0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",ART_AND_DESIGN,4.7,87510,8.7M,"5,000,000+",Free,0,Everyone,Art & Design,"August 1, 2018",1.2.4,4.0.3 and up
3,Sketch - Draw & Paint,ART_AND_DESIGN,4.5,215644,25M,"50,000,000+",Free,0,Teen,Art & Design,"June 8, 2018",Varies with device,4.2 and up
4,Pixel Draw - Number Art Coloring Book,ART_AND_DESIGN,4.3,967,2.8M,"100,000+",Free,0,Everyone,Art & Design;Creativity,"June 20, 2018",1.1,4.4 and up


#1

In [42]:
# ========================
# IMPORT LIBRARIES
# ========================
import pandas as pd
import numpy as np
import plotly.express as px
from datetime import datetime
import pytz

# ========================
# LOAD DATA
# ========================
df = pd.read_csv("Play Store Data.csv")

# ========================
# CLEAN & PREPARE DATA
# ========================

df['Rating'] = pd.to_numeric(df['Rating'], errors='coerce')
df['Reviews'] = pd.to_numeric(df['Reviews'], errors='coerce')

df['Installs'] = (
    df['Installs']
    .astype(str)
    .str.replace('[+,]', '', regex=True)
)
df['Installs'] = pd.to_numeric(df['Installs'], errors='coerce')

def size_to_mb(size):
    try:
        if 'M' in size:
            return float(size.replace('M', ''))
        elif 'k' in size or 'K' in size:
            return float(size.replace('k', '').replace('K', '')) / 1024
    except:
        return None

df['Size_MB'] = df['Size'].astype(str).apply(size_to_mb)

df['Last Updated'] = pd.to_datetime(df['Last Updated'], errors='coerce')

# ========================
# FILTER DATA
# ========================
filtered_df = df[
    (df['Rating'] >= 4.0) &
    (df['Size_MB'] >= 10) &
    (df['Last Updated'].dt.month == 1)
]

# ========================
# TOP 10 CATEGORIES
# ========================
top_categories = (
    filtered_df.groupby('Category')['Installs']
    .sum()
    .nlargest(10)
    .index
)

filtered_df = filtered_df[filtered_df['Category'].isin(top_categories)]

category_summary = (
    filtered_df.groupby('Category')
      .agg(
          Average_Rating=('Rating', 'mean'),
          Total_Reviews=('Reviews', 'sum')
      )
      .reset_index()
)

# ========================
# BAR CHART
# ========================
fig1 = px.bar(
    category_summary,
    x='Category',
    y=['Average_Rating', 'Total_Reviews'],
    barmode='group',
    title='Top 10 Categories: Average Rating vs Reviews',
    color_discrete_map={'Average_Rating':'#1f77b4','Total_Reviews':'#17becf'}
)

fig1.update_layout(
    plot_bgcolor='white',
    paper_bgcolor='white',
    font=dict(color='black')
)

fig1.show()

# ========================
# PIE CHART
# ========================
type_counts = df['Type'].value_counts()

fig2 = px.pie(
    values=type_counts.values,
    names=type_counts.index,
    title="App Type Distribution",
    color_discrete_sequence=['#4C78A8', '#F58518', '#54A24B', '#E45756']
)

fig2.update_layout(
    plot_bgcolor='white',
    paper_bgcolor='white',
    font=dict(color='black')
)

fig2.show()

# ========================
# HISTOGRAM
# ========================
fig3 = px.histogram(
    df,
    x='Rating',
    nbins=20,
    title='Rating Distribution',
    color_discrete_sequence=['#E45756']
)

fig3.update_layout(
    plot_bgcolor='white',
    paper_bgcolor='white',
    font=dict(color='black')
)

fig3.show()


#2

In [50]:
from datetime import datetime
import pytz
import pandas as pd
import plotly.graph_objects as go

# ========================
# LOAD DATA
# ========================
df = pd.read_csv("Play Store Data.csv")

# ========================
# TIME CHECK (1 PM – 2 PM IST)
# ========================
ist = pytz.timezone("Asia/Kolkata")
current_time = datetime.now(ist).time()

show_task2 = (
    current_time >= datetime.strptime("00:00", "%H:%M").time() and
    current_time <= datetime.strptime("23:59", "%H:%M").time()
)

# ========================
# DATA CLEANING
# ========================

# Clean Installs
df['Installs'] = df['Installs'].astype(str).str.replace('[+,]', '', regex=True)
df['Installs'] = pd.to_numeric(df['Installs'], errors='coerce')

# Clean Price
df['Price'] = df['Price'].astype(str).str.replace('$', '', regex=False)
df['Price'] = pd.to_numeric(df['Price'], errors='coerce')

# Convert Size to MB
def size_to_mb(size):
    try:
        if 'M' in size:
            return float(size.replace('M', ''))
        elif 'k' in size or 'K' in size:
            return float(size.replace('k','').replace('K','')) / 1024
    except:
        return None

df['Size_MB'] = df['Size'].astype(str).apply(size_to_mb)

# Android version numeric
df['Android_Num'] = df['Android Ver'].str.extract(r'(\d+\.?\d*)')[0]
df['Android_Num'] = pd.to_numeric(df['Android_Num'], errors='coerce')

# Revenue
df['Revenue'] = df['Price'] * df['Installs']

# Remove invalid rows
df = df.dropna(subset=['Installs','Price','Size_MB','Android_Num','Revenue'])

# ========================
# FILTERS
# ========================
if show_task2:

    df = df[
        (df['Installs'] >= 10000) &
        (df['Revenue'] >= 10000) &
        (df['Android_Num'] > 4.0) &
        (df['Size_MB'] > 15) &
        (df['Content Rating'] == 'Everyone') &
        (df['App'].str.len() <= 30)
    ]

    # Top 3 categories by installs
    top_categories = df.groupby('Category')['Installs'].sum().nlargest(3).index
    df = df[df['Category'].isin(top_categories)]

    # Summary
    summary = df.groupby(['Category','Type']).agg(
        Avg_Installs=('Installs','mean'),
        Total_Revenue=('Revenue','sum')
    ).reset_index()

    summary['Label'] = summary['Category'] + " - " + summary['Type']

    # ========================
    # DUAL AXIS CHART
    # ========================
    fig4 = go.Figure()

    fig4.add_bar(
        x=summary['Label'],
        y=summary['Avg_Installs'],
        name='Average Installs',
        yaxis='y1'
    )

    fig4.add_trace(
        go.Scatter(
            x=summary['Label'],
            y=summary['Total_Revenue'],
            name='Revenue',
            yaxis='y2',
            mode='lines+markers'
        )
    )

    fig4.update_layout(
        title="Average Installs vs Revenue (Free vs Paid Apps)",
        xaxis_title="Category - Type",
        yaxis=dict(title="Average Installs"),
        yaxis2=dict(title="Revenue", overlaying="y", side="right"),
        plot_bgcolor='white',
        paper_bgcolor='white'
    )

    fig4.show()

else:
    print("Graph visible only between 1 PM and 2 PM IST")


#3

In [51]:
# =========================
# TASK 3: INTERACTIVE CHOROPLETH MAP (MORE VISIBLE HIGHLIGHT)
# =========================

import pandas as pd
import plotly.express as px
from datetime import datetime
import pytz

# -------------------------
# Load Dataset
# -------------------------
playstore_df = pd.read_csv("Play Store Data.csv")

# -------------------------
# Check IST Time (6 PM – 8 PM only)
# -------------------------
ist = pytz.timezone("Asia/Kolkata")
current_time = datetime.now(ist).time()

start_time = datetime.strptime("00:00", "%H:%M").time()
end_time   = datetime.strptime("23:59", "%H:%M").time()

if start_time <= current_time <= end_time:

    df = playstore_df.copy()

    # -------------------------
    # Clean Installs Column
    # -------------------------
    df['Installs'] = pd.to_numeric(
        df['Installs'].astype(str).str.replace('[+,]', '', regex=True),
        errors='coerce'
    )
    df = df.dropna(subset=['Installs'])
    df['Installs'] = df['Installs'].astype(int)

    # -------------------------
    # Remove Categories Starting with A, C, G, S
    # -------------------------
    df = df[~df['Category'].str.startswith(('A','C','G','S'))]

    # -------------------------
    # Get Top 5 Categories
    # -------------------------
    top_categories = (
        df.groupby('Category')['Installs']
        .sum()
        .sort_values(ascending=False)
        .head(5)
        .reset_index()
    )

    # -------------------------
    # Simulated Global Mapping
    # -------------------------
    country_codes = [
        'USA','IND','BRA','FRA','DEU','JPN','GBR','CAN','AUS','ITA',
        'ESP','MEX','RUS','CHN','ZAF','SGP','NZL','NOR','SWE'
    ]

    rows = []

    for _, row in top_categories.iterrows():
        for country in country_codes:
            rows.append({
                'Category': row['Category'],
                'Installs': row['Installs'],
                'iso_alpha': country
            })

    choropleth_df = pd.DataFrame(rows)

    # -------------------------
    # Highlight Install > 1 Million
    # -------------------------
    choropleth_df['Highlight'] = choropleth_df['Installs'] > 1_000_000

    # -------------------------
    # Plot Choropleth (Stronger Colors)
    # -------------------------
    fig5 = px.choropleth(
        choropleth_df,
        locations='iso_alpha',
        color='Highlight',
        hover_name='Category',
        hover_data={'Installs': True, 'iso_alpha': False},
        color_discrete_map={
            True: '#ff0000',   # Bright Red (Highlighted)
            False: '#d3d3d3'   # Light Grey
        },
        title='Top 5 App Categories by Total Installs (Highlighted >1M)'
    )

    # Strong borders for visibility
    fig5.update_traces(marker_line_width=1.5, marker_line_color='black')

    fig5.update_layout(
        geo=dict(
            showframe=False,
            showcoastlines=True,
            coastlinecolor="black",
            projection_type='natural earth'
        ),
        legend_title_text="Installs > 1M"
    )

    fig5.show()

else:
    print("Task 3 chart will display only between 6 PM and 8 PM IST.")


#4

In [52]:
# =========================
# TASK 4: STACKED AREA CHART
# =========================
from datetime import datetime
import pytz
import pandas as pd
import plotly.express as px
import numpy as np

# IST timezone
ist = pytz.timezone("Asia/Kolkata")
current_time = datetime.now(ist).time()

# Show only between 4 PM – 6 PM IST
show_task4 = (
    current_time >= datetime.strptime("00:00", "%H:%M").time() and
    current_time <= datetime.strptime("23:59", "%H:%M").time()
)

if show_task4:

    df4 = playstore_df.copy()

    # -------------------------
    # Data Cleaning
    # -------------------------
    df4['Rating'] = pd.to_numeric(df4['Rating'], errors='coerce')
    df4['Reviews'] = pd.to_numeric(df4['Reviews'], errors='coerce')
    df4['Installs'] = pd.to_numeric(
        df4['Installs'].astype(str).str.replace('[+,]', '', regex=True),
        errors='coerce'
    )

    # Convert Size to MB
    def size_to_mb(size):
        if isinstance(size, str):
            if size.endswith('k'):
                return float(size[:-1]) / 1024
            elif size.endswith('M'):
                return float(size[:-1])
        return np.nan

    df4['Size_MB'] = df4['Size'].apply(size_to_mb)

    # -------------------------
    # Filters
    # -------------------------
    df4 = df4[
        (df4['Rating'] >= 4.2) &
        (df4['Reviews'] > 1000) &
        (~df4['App'].str.contains(r'\d', na=False)) &
        (df4['Category'].str.startswith(('T','P'))) &
        (df4['Size_MB'].between(20,80)) &
        (df4['Installs'].notna())
    ]

    # -------------------------
    # Translate legend names
    # -------------------------
    category_translation = {
        'Travel & Local': 'Voyage et Local',   # French
        'Productivity': 'Productividad',       # Spanish
        'Photography': '写真'                   # Japanese
    }
    df4['Category'] = df4['Category'].replace(category_translation)

    # -------------------------
    # Time processing
    # -------------------------
    df4['Last Updated'] = pd.to_datetime(df4['Last Updated'], errors='coerce')
    df4 = df4.dropna(subset=['Last Updated'])

    df4['Month'] = df4['Last Updated'].dt.to_period('M').dt.to_timestamp()

    # -------------------------
    # Aggregation
    # -------------------------
    df4 = df4.groupby(['Month','Category'], as_index=False)['Installs'].sum()

    df4['Cumulative_Installs'] = df4.groupby('Category')['Installs'].cumsum()

    # -------------------------
    # Month-over-month growth detection
    # -------------------------
    monthly_total = df4.groupby('Month')['Installs'].sum().reset_index()
    monthly_total['Growth'] = monthly_total['Installs'].pct_change()*100
    highlight_months = monthly_total[monthly_total['Growth'] > 25]['Month']

    # -------------------------
    # Plot
    # -------------------------
    fig6 = px.area(
        df4,
        x='Month',
        y='Cumulative_Installs',
        color='Category',
        title="Cumulative Installs Over Time by Category",
        template="plotly_white"
    )

    # Highlight months where growth >25%
    for m in highlight_months:
        fig4.add_vrect(
            x0=m,
            x1=m + pd.offsets.MonthEnd(1),
            fillcolor="red",
            opacity=0.15,
            line_width=0
        )

    fig6.update_layout(
        xaxis_title="Month",
        yaxis_title="Cumulative Installs",
        legend_title="Category"
    )

    fig6.show()

else:
    print("Task 4 visualization available only between 4 PM and 6 PM IST.")


#5

In [53]:
# =========================================
# TASK 5 – FINAL JUPYTER NOTEBOOK VERSION
# =========================================

import pandas as pd
import numpy as np
import plotly.express as px
from textblob import TextBlob
from datetime import datetime
import pytz

# ---------------------------
# LOAD DATA
# ---------------------------
apps_df = pd.read_csv("Play Store Data.csv")
reviews_df = pd.read_csv("User Reviews.csv")

# ---------------------------
# CLEANING
# ---------------------------
apps_df = apps_df.dropna(subset=['Rating'])
apps_df = apps_df[apps_df['Rating'] <= 5]

reviews_df = reviews_df.dropna(subset=['Translated_Review'])

apps_df['Installs'] = (
    apps_df['Installs']
    .str.replace(',', '')
    .str.replace('+', '')
    .astype(int)
)

apps_df['Reviews'] = apps_df['Reviews'].astype(int)

# ---------------------------
# SIZE TO MB
# ---------------------------
def convert_size(size):
    if isinstance(size, str):
        if 'M' in size:
            return float(size.replace('M',''))
        elif 'K' in size:
            return float(size.replace('K',''))/1024
    return np.nan

apps_df['Size_MB'] = apps_df['Size'].apply(convert_size)

# ---------------------------
# SENTIMENT SUBJECTIVITY
# ---------------------------
reviews_df['Subjectivity'] = reviews_df['Translated_Review'].apply(
    lambda x: TextBlob(str(x)).sentiment.subjectivity
)

# ---------------------------
# MERGE
# ---------------------------
merged_df = pd.merge(apps_df, reviews_df, on='App', how='inner')

merged_df['Category_Upper'] = merged_df['Category'].str.upper()

# ---------------------------
# FILTER CONDITIONS
# ---------------------------
allowed_categories = [
    'GAME','BEAUTY','BUSINESS','COMICS','COMMUNICATION',
    'DATING','ENTERTAINMENT','SOCIAL','EVENTS'
]

filtered_df = merged_df[
    (merged_df['Rating'] > 3.5) &
    (merged_df['Category_Upper'].isin(allowed_categories)) &
    (merged_df['Reviews'] > 500) &
    (merged_df['Installs'] > 50000) &
    (merged_df['Subjectivity'] > 0.5)
]

# Remove apps containing letter S
filtered_df = filtered_df[~filtered_df['App'].str.contains('S', case=False)]

filtered_df = filtered_df.dropna(subset=['Size_MB'])

# ---------------------------
# TRANSLATE CATEGORY DISPLAY
# ---------------------------
translation = {
    'BEAUTY':'सौंदर्य',        # Hindi
    'BUSINESS':'வணிகம்',      # Tamil
    'DATING':'Partnersuche'   # German
}

filtered_df['Category_Display'] = filtered_df['Category_Upper'].replace(translation)

# ---------------------------
# TIME CHECK (5 PM – 7 PM IST)
# ---------------------------
IST = pytz.timezone("Asia/Kolkata")
current_time = datetime.now(IST)

if 00 <= current_time.hour < 24 and not filtered_df.empty:

    fig7 = px.scatter(
        filtered_df,
        x='Size_MB',
        y='Rating',
        size='Installs',
        color='Category_Display',
        hover_name='App',
        size_max=60,
        title='App Size vs Rating Bubble Chart'
    )

    # Highlight GAME category in pink
    game_df = filtered_df[filtered_df['Category_Upper']=='GAME']
    if not game_df.empty:
        fig_game = px.scatter(
            game_df,
            x='Size_MB',
            y='Rating',
            size='Installs',
            hover_name='App'
        )
        for t in fig_game.data:
            t.marker.color = 'pink'
            t.marker.line.width = 2
            t.marker.line.color = 'white'
            fig.add_trace(t)

    fig7.show()

else:
    print("Graph not shown (outside 5PM–7PM IST or no data after filtering)")



#6

In [54]:
# ===============================
# TASK 6 – ENHANCED VISUAL VERSION
# ===============================

import pandas as pd
import plotly.graph_objects as go
from datetime import datetime
import pytz

# ---- TIME CHECK (6 PM – 9 PM IST) ----
ist = pytz.timezone("Asia/Kolkata")
current_time = datetime.now(ist).time()

if datetime.strptime("00:00","%H:%M").time() <= current_time <= datetime.strptime("23:59","%H:%M").time():

    df6 = playstore_df.copy()

    # ---- CLEAN NUMERIC ----
    df6['Installs'] = (
        df6['Installs']
        .astype(str)
        .str.replace('[+,]', '', regex=True)
    )
    df6['Installs'] = pd.to_numeric(df6['Installs'], errors='coerce')
    df6['Reviews'] = pd.to_numeric(df6['Reviews'], errors='coerce')

    df6 = df6.dropna(subset=['Installs','Reviews'])

    # ---- APPLY FILTERS ----
    df6 = df6[
        (df6['Reviews'] > 500) &
        (~df6['App'].str.contains('S', case=False, na=False)) &
        (~df6['App'].str.startswith(('x','y','z'), na=False)) &
        (df6['Category'].str.startswith(('E','C','B'), na=False))
    ]

    # ---- DATE PREP ----
    df6['Last Updated'] = pd.to_datetime(df6['Last Updated'], errors='coerce')
    df6 = df6.dropna(subset=['Last Updated'])

    df6['Month'] = df6['Last Updated'].dt.to_period('M').astype(str)

    # ---- GROUP ----
    df_group = df6.groupby(['Month','Category'])['Installs'].sum().reset_index()

    # ---- MOM GROWTH ----
    df_group['MoM_Growth'] = (
        df_group.groupby('Category')['Installs']
        .pct_change()
    )

    df_group['Highlight'] = df_group['MoM_Growth'] > 0.20

    # ---- TRANSLATION FIX (UPPERCASE MATCH) ----
    translation_map = {
        'BEAUTY': 'सौंदर्य',        # Hindi
        'BUSINESS': 'வணிகம்',      # Tamil
        'DATING': 'Partnersuche'   # German
    }

    df_group['Category_Display'] = df_group['Category'].replace(translation_map)

    # ---- SORT MONTH ----
    df_group = df_group.sort_values("Month")

    # ---- PLOT ----
    fig8 = go.Figure()

    for cat in df_group['Category_Display'].unique():

        cat_df = df_group[df_group['Category_Display']==cat]

        # Smooth professional line
        fig8.add_trace(go.Scatter(
            x=cat_df['Month'],
            y=cat_df['Installs'],
            mode='lines+markers',
            name=cat,
            line=dict(width=3, shape='spline'),   # smooth curve
            marker=dict(size=6)
        ))

        # Highlight growth >20%
        highlight_df = cat_df[cat_df['Highlight']]

        fig8.add_trace(go.Scatter(
            x=highlight_df['Month'],
            y=highlight_df['Installs'],
            fill='tozeroy',
            opacity=0.18,
            mode='lines',
            showlegend=False
        ))

    fig8.update_layout(
        title="Total Installs Over Time by Category",
        xaxis_title="Month",
        yaxis_title="Total Installs",
        template="plotly_white",
        height=600,
        legend_title="Category"
    )

    fig8.show()

else:
    print("Task 6 visible only between 6 PM and 9 PM IST")


#dashboard

In [57]:
import plotly.offline as pyo
import webbrowser
import os

# Function to convert figure to HTML div
def get_plot_div(fig):
    return pyo.plot(fig, include_plotlyjs='cdn', output_type='div')

plot_containers = ""

tasks = [
    ('fig1',"Task 1: Bar Chart"),
    ('fig2',"Task 1: Pie Chart"),
    ('fig3',"Task 1: Histogram"),
    ('fig4',"Task 2: Dual Axis Analysis"),
    ('fig5',"Task 3: Choropleth Map"),
    ('fig6',"Task 4: Area Chart"),
    ('fig7',"Task 5: Bubble Chart"),
    ('fig8',"Task 6: Line Trend")
]

for var_name, title in tasks:
    if var_name in globals():
        plot_div = get_plot_div(globals()[var_name])
        plot_containers += f"""
        <div class="card">
            <h2>{title}</h2>
            {plot_div}
        </div>
        """

dashboard_html = f"""
<!DOCTYPE html>
<html>
<head>
<meta charset="UTF-8">
<title>Google Play Store Reviews Analytics</title>

<style>
body {{
    margin:0;
    font-family: Arial, Helvetica, sans-serif;
    background: linear-gradient(135deg,#141e30,#243b55);
    color:white;
}}

.header {{
    background: linear-gradient(90deg,#0f2027,#203a43,#2c5364);
    padding:20px 40px;
}}

.header-content {{
    display:flex;
    justify-content:space-between;
    align-items:center;
}}

.header h1 {{
    margin:0;
}}

.logo {{
    height:50px;
}}

.dashboard {{
    display:grid;
    grid-template-columns:1fr 1fr;
    gap:25px;
    padding:40px;
}}

.card {{
    background:rgba(255,255,255,0.08);
    border-radius:20px;
    padding:20px;
    box-shadow:0 4px 20px rgba(0,0,0,0.4);
}}

.card h2 {{
    text-align:center;
    margin-bottom:15px;
}}
</style>
</head>

<body>

<div class="header">
    <div class="header-content">
        <h1>Google Play Store Reviews Analytics Dashboard</h1>
        <img src="https://upload.wikimedia.org/wikipedia/commons/7/78/Google_Play_Store_badge_EN.svg" class="logo">
    </div>
</div>

<div class="dashboard">
{plot_containers}
</div>

</body>
</html>
"""

# Save file
output_path = os.path.abspath("dashboard.html")
with open(output_path, "w", encoding="utf-8") as f:
    f.write(dashboard_html)

# Open in browser
webbrowser.open('file://' + output_path)


True